In [6]:
import time
def fake_download(name,seconds):
    print(f"  ⬇️  开始下载 {name}...")
    time.sleep(seconds)
    print(f"  ✅ {name} 下载完成！")
    return f'{name}:{seconds}'
files = [("report.pdf", 2), ("video.mp4", 3), ("music.mp3", 1), ("image.png", 2)]
print("=== 串行下载 ===")
results = []
start = time.perf_counter()
for name,sec in files:
    results.append(fake_download(name,sec))
    serial_time = time.perf_counter() - start
print(f"总耗时: {serial_time:.1f}s\n")

=== 串行下载 ===
  ⬇️  开始下载 report.pdf...
  ✅ report.pdf 下载完成！
  ⬇️  开始下载 video.mp4...
  ✅ video.mp4 下载完成！
  ⬇️  开始下载 music.mp3...
  ✅ music.mp3 下载完成！
  ⬇️  开始下载 image.png...
  ✅ image.png 下载完成！
总耗时: 8.0s



In [18]:
import time
from concurrent.futures import ThreadPoolExecutor
def fake_download(name,seconds):
    print(f"  ⬇️  开始下载 {name}...")
    time.sleep(seconds)
    print(f"  ✅ {name} 下载完成！")
    return f"{name} ({seconds}s)"
files = [("report.pdf", 2), ("video.mp4", 3), ("music.mp3", 1), ("image.png", 2)]
print("=== 并发下载 ===")
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = []
    for name,sec in files:
        future = executor.submit(fake_download,name,sec)
        futures.append(future)

    results = [f.result() for f in futures]
parallel_time = time.perf_counter() - start
print(results)
print(f"总耗时: {parallel_time:.1f}s")

=== 并发下载 ===
  ⬇️  开始下载 report.pdf...
  ⬇️  开始下载 video.mp4...
  ⬇️  开始下载 music.mp3...
  ⬇️  开始下载 image.png...
  ✅ music.mp3 下载完成！
  ✅ report.pdf 下载完成！  ✅ image.png 下载完成！

  ✅ video.mp4 下载完成！
['report.pdf (2s)', 'video.mp4 (3s)', 'music.mp3 (1s)', 'image.png (2s)']
总耗时: 3.0s


In [35]:
import time
from concurrent.futures import ThreadPoolExecutor
def fetch_url(url):
    delay = len(url) % 3 + 1
    time.sleep(delay)
    return f"{url} → 200 OK ({delay}s)"
urls = [
    "https://api.example.com/users",
    "https://api.example.com/products",
    "https://api.example.com/orders",
    "https://api.example.com/reviews",
    "https://api.example.com/comments",
]
print("=== executor.map ===")
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
        results = executor.map(fetch_url,urls)
        for result in results:
          print(f"  {result}")
elapsed = time.perf_counter() - start
print(f"\n总耗时: {elapsed:.1f}s(串行需要 {sum(len(u)%3+1 for u in urls)})")
def download(url,timeout):
    time.sleep(timeout)
    return f"{url} (timeout={timeout})"
tasks = [("url1", 1), ("url2", 2), ("url3", 1)]
with ThreadPoolExecutor(max_workers=3) as executor:
    results = list(executor.map(lambda t:download(*t),tasks))
    print(f"\n多参数: {results}")

=== executor.map ===
  https://api.example.com/users → 200 OK (3s)
  https://api.example.com/products → 200 OK (3s)
  https://api.example.com/orders → 200 OK (1s)
  https://api.example.com/reviews → 200 OK (2s)
  https://api.example.com/comments → 200 OK (3s)

总耗时: 3.0s(串行需要 12)

多参数: ['url1 (timeout=1)', 'url2 (timeout=2)', 'url3 (timeout=1)']


In [42]:
#导入与模拟任务
import time
import random
from concurrent.futures import ThreadPoolExecutor,as_completed
def slow_task(name):
    delay = random.uniform(0.5,3.0)
    time.sleep(delay)
    return f"{name} 完成(耗时 {delay:.1f}s)"
#基础示例：谁先完成谁输出
tasks = ["任务A", "任务B", "任务C", "任务D", "任务E"]
print("=== as_completed:谁先完成谁先输出 ===\n")
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    future_to_name = {
             executor.submit(slow_task,name):name 
             for name in tasks
        }
    for future in as_completed(future_to_name):
        name = future_to_name[future]
        try:
            result = future.result()
            print(f"  ✅ {result}")
        except Exception as e:
            print(f"  ❌ {name} 失败: {e}")
elapsed = time.perf_counter() - start 
print(f"\n总耗时: {elapsed:.1f}s")
#带超时的as_completed
print("\n=== 带超时的 as_completed ===")
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(slow_task,name):name for name in tasks}
    for future in as_completed(futures,timeout=2.0):
        name = futures[future]
        try:
            print(f"  ✅ {future.result()}")
        except Exception as e:
            print(f"  ❌ {name}: {e}")


=== as_completed:谁先完成谁先输出 ===

  ✅ 任务E 完成(耗时 0.6s)
  ✅ 任务B 完成(耗时 1.3s)
  ✅ 任务C 完成(耗时 1.4s)
  ✅ 任务A 完成(耗时 1.7s)
  ✅ 任务D 完成(耗时 2.4s)

总耗时: 2.4s

=== 带超时的 as_completed ===
  ✅ 任务B 完成(耗时 0.6s)
  ✅ 任务C 完成(耗时 1.6s)
  ✅ 任务A 完成(耗时 1.9s)
  ✅ 任务D 完成(耗时 1.3s)


TimeoutError: 1 (of 5) futures unfinished

In [ ]:
from concurrent.futures import ThreadPoolExecutor,as_completed
def risky_task(n):
    if n == 3:
        raise ValueError(f"数字 {n} 不吉利！")
    if n == 5:
        raise ConnectionError("网络断了")
    return n * n
print("=== 正确的异常处理 ===\n")
with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {executor.submit(risky_task,i):i for i in range(7)}
    for future in as_completed(futures):
        n = futures[future]
        try:
            result = future.result()#抛出异常
            print(f"  ✅ task({n}) = {result}")
        except ValueError as e:
            print(f"  ⚠️  task({n}) 值错误: {e}")
        except ConnectionError as e:
            print(f"  ❌ task({n}) 连接错误: {e}")


=== 正确的异常处理 ===

  ❌ task(5) 连接错误: 网络断了
  ⚠️  task(3) 值错误: 数字 3 不吉利！
  ✅ task(0) = 0
  ✅ task(1) = 1
  ✅ task(4) = 16
  ✅ task(2) = 4
  ✅ task(6) = 36


In [96]:
import threading
from concurrent.futures import ThreadPoolExecutor
counter_unsafe = 0
lock = threading.Lock()
def increment_safe(n):
    global counter_safe
    for _ in range(n):
        with lock:
            counter_safe += 1
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(increment_safe,100000) for _ in range(10)]
    for f in futures:
        f.result()
print(f"✅ 加锁后: {counter_safe}（一定是 1,000,000）")

✅ 加锁后: 22000000（一定是 1,000,000）
